# Create Ahmanson Foundation Awards

Creates OpenAlex award rows from The Ahmanson Foundation's official grants archive.

**Awarding body:** Ahmanson Foundation - F4320314405 (US; no ROR or DOI in OpenAlex)

**Source:** https://theahmansonfoundation.org/grants/

**Source method:** official WordPress `fc_grants` archive rendered as static HTML. The page exposes one `article.fc-grant` per grant with native WordPress post IDs, grantee names, program areas, fiscal years, locations, purposes, and dollar amounts.

**Local validation on 2026-05-28:** 7,727 rows, 7,727 unique native IDs, fiscal years 2010-2025, 100% title/grantee/program/year/landing-page coverage, 7,726/7,727 amount and currency coverage, 99.9% recipient country coverage, 7,724/7,727 description coverage, total USD 844,169,563.

**S3 staging path:** `s3a://openalex-ingest/awards/ahmanson/ahmanson_grants.parquet`

**Schema choices:**
- `funder_award_id` is `ahmanson-{post_id}`, using the source's stable WordPress post ID.
- `funding_type` is `grant` for all rows.
- `funder_scheme` is the source program area.
- `start_date` is fiscal-year January 1 because the source publishes fiscal years, not exact award dates.
- `lead_investigator` is modeled as an organization-only recipient using the source grantee name and location-derived US country code where available.
- `amount` and `currency` are populated directly from source dollar amounts; one official row lacks an amount and remains NULL.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ahmanson_grants_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ahmanson/ahmanson_grants.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 7,727 rows.
SELECT COUNT(*) AS total_ahmanson_grants
FROM openalex.awards.ahmanson_grants_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.ahmanson_grants_raw;


In [ ]:
%sql
-- Sample raw Ahmanson data.
SELECT
    funder_award_id,
    post_id,
    display_name,
    grantee_name,
    amount,
    currency,
    program_area,
    fiscal_year,
    recipient_location,
    recipient_country,
    landing_page_url
FROM openalex.awards.ahmanson_grants_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Do not wrap DESCRIBE in a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'ahmanson_grants_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|total|funding|award|grant|currency|usd|dollar|value|budget|cost'
ORDER BY ordinal_position;


In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 2) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 2) AS pct_with_currency,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_usd,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_usd,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_usd,
    MIN(TRY_CAST(fiscal_year AS INT)) AS min_fiscal_year,
    MAX(TRY_CAST(fiscal_year AS INT)) AS max_fiscal_year
FROM openalex.awards.ahmanson_grants_raw;


In [ ]:
%sql
-- Native-key inspection. Native WordPress post IDs are stable and unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT post_id) AS distinct_post_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.ahmanson_grants_raw;


In [ ]:
%sql
-- Source program and fiscal-year distribution.
SELECT
    fiscal_year,
    program_area,
    COUNT(*) AS rows,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_usd
FROM openalex.awards.ahmanson_grants_raw
GROUP BY fiscal_year, program_area
ORDER BY TRY_CAST(fiscal_year AS INT) DESC, rows DESC;


In [ ]:
%sql
-- Recipient location coverage.
SELECT
    COUNT(*) AS total,
    COUNT(recipient_location) AS has_location,
    ROUND(try_divide(COUNT(recipient_location), COUNT(*)) * 100.0, 2) AS pct_location,
    COUNT(recipient_country) AS has_country,
    ROUND(try_divide(COUNT(recipient_country), COUNT(*)) * 100.0, 2) AS pct_country,
    collect_set(recipient_country) AS countries
FROM openalex.awards.ahmanson_grants_raw;


## Step 1.6: Funder Existence Fail-Fast

Runbook Step 2.2.4 mandatory check. Must return exactly 1 row for Ahmanson Foundation.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320314405;


In [ ]:
%sql
-- Fail-fast count form. Expected: exactly one row.
SELECT COUNT(*) AS funder_rows
FROM openalex.common.funder
WHERE funder_id = 4320314405;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ahmanson_awards
USING delta
AS
WITH
ahmanson_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314405
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'USD' ELSE NULL END AS parsed_currency,
        TRY_CAST(fiscal_year AS INT) AS parsed_year,
        CASE
            WHEN TRY_CAST(fiscal_year AS INT) BETWEEN 1900 AND YEAR(current_date()) + 1
            THEN TRY_TO_DATE(CONCAT(fiscal_year, '-01-01'), 'yyyy-MM-dd')
            ELSE NULL
        END AS parsed_start_date
    FROM openalex.awards.ahmanson_grants_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        TRIM(g.display_name) AS display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        g.parsed_amount AS amount,
        g.parsed_currency AS currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        'grant' AS funding_type,

        NULLIF(TRIM(g.program_area), '') AS funder_scheme,

        'ahmanson_grants_archive' AS provenance,

        g.parsed_start_date AS start_date,
        CAST(NULL AS DATE) AS end_date,
        g.parsed_year AS start_year,
        CAST(NULL AS INT) AS end_year,

        struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            g.parsed_start_date AS role_start,
            struct(
                NULLIF(TRIM(g.grantee_name), '') AS name,
                NULLIF(TRIM(g.recipient_country), '') AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        CONCAT('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date

    FROM raw_prepared g
    CROSS JOIN ahmanson_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 166

In [ ]:
%sql
-- Remove previous Ahmanson archive data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ahmanson_grants_archive' AND priority = 166;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    166 AS priority
FROM openalex.awards.ahmanson_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_ahmanson_awards
FROM openalex.awards.ahmanson_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.ahmanson_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    lead_investigator.affiliation.name AS recipient_org,
    lead_investigator.affiliation.country AS recipient_country,
    landing_page_url
FROM openalex.awards.ahmanson_awards
LIMIT 10;


In [ ]:
%sql
-- Check OpenAlex ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.ahmanson_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only Ahmanson Foundation.
SELECT funder.display_name, funder_id, COUNT(*) AS rows
FROM openalex.awards.ahmanson_awards
GROUP BY funder.display_name, funder_id
ORDER BY rows DESC;


In [ ]:
%sql
-- Data completeness. Local validation found one official row without amount/currency.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_year) AS has_start_year,
    COUNT(landing_page_url) AS has_url,
    COUNT(lead_investigator.affiliation.name) AS has_recipient_org,
    COUNT(lead_investigator.affiliation.country) AS has_recipient_country,
    ROUND(SUM(amount), 0) AS total_usd,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 2) AS pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 2) AS pct_description,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.country), COUNT(*)) * 100.0, 2) AS pct_recipient_country
FROM openalex.awards.ahmanson_awards;


In [ ]:
%sql
-- Amount/currency fail-fast check for this grant pattern.
-- Expected: 7,726 of 7,727 rows have positive USD amounts; one official source row lacks amount.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 2) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_positive_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.ahmanson_awards;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.ahmanson_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Program distribution.
SELECT funding_type, funder_scheme, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.ahmanson_awards
GROUP BY funding_type, funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 166.
SELECT
    priority,
    provenance,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ahmanson_grants_archive' AND priority = 166
GROUP BY priority, provenance;


## Handoff Notes

Contractor has no S3 or Databricks access. Admin should:

1. Run `scripts/local/ahmanson_to_s3.py --skip-download` without `--skip-upload` after reviewing the local parquet.
2. Run this notebook on Databricks.
3. Inspect Step 6 verification cells for row count, uniqueness, funder mapping, amount/currency coverage, and priority 166 insertion.
4. Only add scheduled job YAML after upload, notebook execution, and QA pass.
